# NBA Game Outcome Prediction
## Notebook 2 — Model Improvements

Building on **Notebook 1** (best result: 63.1% leaky ensemble), this notebook adds five concrete improvements one by one:

| Step | Improvement | Why |
|------|-------------|-----|
| 1 | Elo ratings | Rolling win rate ignores opponent strength |
| 2 | Rest days + back-to-back | Fatigue is a measurable, known effect |
| 3 | Optuna-tuned XGBoost | Default params are not optimal |
| 4 | Attention-LSTM + new features | Recency matters — not all 10 games equally |
| 5 | Proper OOF stacking | Notebook 1 stacking had data leakage |

**Notebook 1 best:** 63.1% &nbsp;|&nbsp; **Target:** 70%

In [1]:
import pandas as pd
import numpy as np
import kagglehub
import os
import warnings
import importlib
import subprocess
import sys
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, Layer
from tensorflow.keras.callbacks import EarlyStopping

print('All imports successful')
print(f'TensorFlow: {tf.__version__}')

2026-04-22 19:11:07.898748: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


All imports successful
TensorFlow: 2.16.2


## Base Setup — Recreate Everything from Notebook 1

This cell re-runs the exact same data pipeline as Notebook 1 so this notebook is fully self-contained. Nothing new here — just the foundation.

In [2]:
# ── LOAD DATA ───────────────────────────────────────────────────────────────────
path = kagglehub.dataset_download('eoinamoore/historical-nba-data-and-player-box-scores')
games = pd.read_csv(os.path.join(path, 'Games.csv'))

games['gameDateTimeEst'] = pd.to_datetime(games['gameDateTimeEst'])
games = games.sort_values('gameDateTimeEst').reset_index(drop=True)
games = games[games['gameType'] == 'Regular Season'].reset_index(drop=True)
games = games[games['gameDateTimeEst'].dt.year >= 2000].reset_index(drop=True)
games['home_win'] = (games['winner'] == games['hometeamId']).astype(int)

# Team name lookup (teamId -> 'City Name')
home_teams = games[['hometeamId', 'hometeamCity', 'hometeamName']].copy()
home_teams.columns = ['teamId', 'city', 'name']
away_teams = games[['awayteamId', 'awayteamCity', 'awayteamName']].copy()
away_teams.columns = ['teamId', 'city', 'name']
team_lookup = pd.concat([home_teams, away_teams]).drop_duplicates('teamId')
team_lookup['teamName'] = team_lookup['city'] + ' ' + team_lookup['name']
team_lookup = team_lookup[['teamId', 'teamName']]

# Build per-team-per-game table (63,696 rows = 31,848 games x 2 teams)
home_tg = games[['gameId', 'gameDateTimeEst', 'hometeamId', 'homeScore', 'awayScore', 'home_win']].copy()
home_tg.columns = ['gameId', 'date', 'teamId', 'teamScore', 'oppScore', 'win']
away_tg = games[['gameId', 'gameDateTimeEst', 'awayteamId', 'awayScore', 'homeScore', 'home_win']].copy()
away_tg.columns = ['gameId', 'date', 'teamId', 'teamScore', 'oppScore', 'win']
away_tg['win'] = 1 - away_tg['win']

team_games = pd.concat([home_tg, away_tg]).sort_values('date').reset_index(drop=True)
team_games = team_games.merge(team_lookup, on='teamId', how='left')
team_games = team_games.sort_values(['teamName', 'date']).reset_index(drop=True)

# Rolling features (shift(1) on every feature = no data leakage)
team_games['rolling_win_rate'] = team_games.groupby('teamName')['win'].transform(
    lambda x: x.shift(1).rolling(10, min_periods=3).mean())
team_games['rolling_pts_scored'] = team_games.groupby('teamName')['teamScore'].transform(
    lambda x: x.shift(1).rolling(10, min_periods=3).mean())
team_games['rolling_pts_allowed'] = team_games.groupby('teamName')['oppScore'].transform(
    lambda x: x.shift(1).rolling(10, min_periods=3).mean())
team_games['point_diff'] = team_games['teamScore'] - team_games['oppScore']
team_games['rolling_point_diff'] = team_games.groupby('teamName')['point_diff'].transform(
    lambda x: x.shift(1).rolling(10, min_periods=3).mean())

def get_streak(series):
    streaks, current = [], 0
    for val in series:
        if val == 1: current += 1
        else: current = 0
        streaks.append(current)
    return pd.Series(streaks, index=series.index).shift(1).fillna(0)

team_games['win_streak'] = team_games.groupby('teamName')['win'].transform(get_streak)

print(f'team_games shape: {team_games.shape}')
n_teams = team_games['teamName'].nunique()
print(f'Teams: {n_teams}')
print(f'Date range: {team_games["date"].min()} to {team_games["date"].max()}')
print('\nBase features per team row:')
print(team_games.columns.tolist())

team_games shape: (63902, 13)
Teams: 30
Date range: 2000-01-02 19:30:00 to 2026-04-12 20:30:00

Base features per team row:
['gameId', 'date', 'teamId', 'teamScore', 'oppScore', 'win', 'teamName', 'rolling_win_rate', 'rolling_pts_scored', 'rolling_pts_allowed', 'point_diff', 'rolling_point_diff', 'win_streak']


---
## Section 1 — Elo Ratings

**Problem with rolling win rate:** A team that went 7-3 beating playoff teams should rank higher than one that went 7-3 beating lottery teams. Rolling win rate cannot distinguish these.

**Elo fix:** A running strength score that:
- Starts every team at 1500
- Increases more when you beat a strong opponent
- Decreases more when you lose to a weak opponent
- Uses the logistic formula: `E = 1 / (1 + 10^((opp_elo - team_elo) / 400))`

**No leakage:** Elo is recorded **before** the game result is applied — exactly what a bettor would know at tip-off.

In [3]:
# ── IMPROVEMENT 1: ELO RATINGS ──────────────────────────────────────────────────
K_FACTOR    = 20.0
INITIAL_ELO = 1500.0

# Initialize all teams
all_home = games['hometeamCity'] + ' ' + games['hometeamName']
all_away = games['awayteamCity'] + ' ' + games['awayteamName']
all_team_names = pd.concat([all_home, all_away]).unique()
elo_ratings = {t: INITIAL_ELO for t in all_team_names}

elo_records = []

for _, row in games.sort_values('gameDateTimeEst').iterrows():
    h = row['hometeamCity'] + ' ' + row['hometeamName']
    a = row['awayteamCity'] + ' ' + row['awayteamName']

    elo_h = elo_ratings[h]
    elo_a = elo_ratings[a]

    # Expected win probability (pre-game — zero leakage)
    exp_h = 1.0 / (1.0 + 10.0 ** ((elo_a - elo_h) / 400.0))
    exp_a = 1.0 - exp_h

    # Record BEFORE updating
    elo_records.append({
        'gameId':       row['gameId'],
        'home_elo':     round(elo_h, 2),
        'away_elo':     round(elo_a, 2),
        'elo_diff':     round(elo_h - elo_a, 2),   # positive = home favored
        'elo_win_prob': round(exp_h, 4)             # Elo's own probability
    })

    # Update after result
    outcome = row['home_win']
    elo_ratings[h] += K_FACTOR * (outcome - exp_h)
    elo_ratings[a] += K_FACTOR * ((1 - outcome) - exp_a)

elo_df = pd.DataFrame(elo_records)

print(f'Elo records computed: {len(elo_df)}')
print(f'Elo range — min: {elo_df["home_elo"].min():.1f}, max: {elo_df["home_elo"].max():.1f}')

# Add team-level elo to team_games so it can be used in LSTM sequences
home_elo_map = games[['gameId']].copy()
home_elo_map['teamName'] = games['hometeamCity'] + ' ' + games['hometeamName']
home_elo_map = home_elo_map.merge(elo_df[['gameId', 'home_elo']], on='gameId')
home_elo_map = home_elo_map.rename(columns={'home_elo': 'team_elo'})

away_elo_map = games[['gameId']].copy()
away_elo_map['teamName'] = games['awayteamCity'] + ' ' + games['awayteamName']
away_elo_map = away_elo_map.merge(elo_df[['gameId', 'away_elo']], on='gameId')
away_elo_map = away_elo_map.rename(columns={'away_elo': 'team_elo'})

elo_tg = pd.concat([home_elo_map[['gameId', 'teamName', 'team_elo']],
                    away_elo_map[['gameId', 'teamName', 'team_elo']]])

team_games = team_games.merge(elo_tg, on=['gameId', 'teamName'], how='left')

print(f'team_elo added to team_games — nulls: {team_games["team_elo"].isna().sum()}')

# Sanity check: top 5 teams by current Elo
final_elo_series = pd.Series(elo_ratings).sort_values(ascending=False)
print('\nTop 5 teams by final Elo rating:')
for name, elo in final_elo_series.head(5).items():
    print(f'  {name:<35} {elo:.1f}')

print('\nSample elo_df (first 5 rows):')
print(elo_df.head(5).to_string())

Elo records computed: 31951
Elo range — min: 1180.9, max: 1855.2
team_elo added to team_games — nulls: 6661

Top 5 teams by final Elo rating:
  Oklahoma City Thunder               1753.6
  San Antonio Spurs                   1713.3
  Boston Celtics                      1692.9
  Denver Nuggets                      1667.8
  Detroit Pistons                     1667.6

Sample elo_df (first 5 rows):
     gameId  home_elo  away_elo  elo_diff  elo_win_prob
0  29900423    1500.0    1500.0       0.0        0.5000
1  29900425    1500.0    1500.0       0.0        0.5000
2  29900426    1500.0    1500.0       0.0        0.5000
3  29900424    1500.0    1500.0       0.0        0.5000
4  29900427    1490.0    1500.0     -10.0        0.4856


---
## Section 2 — Rest Days + Back-to-Back Games

**Why it matters:** Playing on the second night of a back-to-back (zero days rest) is one of the most consistent, well-documented performance penalties in NBA analytics. Teams on B2B games win roughly 3–4% less often than their rolling stats predict.

**Implementation:** For each team in `team_games`, compute the gap in days between consecutive games. The first game of a season gets 7 days rest (safe prior — no performance penalty).

**No leakage:** The rest period before a game is fully known at tip-off.

In [4]:
# ── IMPROVEMENT 2: REST DAYS + BACK-TO-BACK ─────────────────────────────────────
team_games = team_games.sort_values(['teamName', 'date']).reset_index(drop=True)

team_games['days_rest'] = (
    team_games.groupby('teamName')['date']
    .diff()
    .dt.total_seconds()
    .div(86400)       # seconds -> days
    .clip(upper=14)   # cap at 14 (start of season / all-star break look the same)
    .fillna(7)        # first game each season: assume full rest
)

team_games['is_back_to_back'] = (team_games['days_rest'] <= 1).astype(int)

# Verify the effect is real
b2b_wr    = team_games[team_games['is_back_to_back'] == 1]['win'].mean()
normal_wr = team_games[team_games['is_back_to_back'] == 0]['win'].mean()
n_b2b     = team_games['is_back_to_back'].sum()

print('Rest Days + Back-to-Back computed')
print(f'\nB2B games in dataset:    {n_b2b:,}')
print(f'B2B win rate:            {b2b_wr:.3f}')
print(f'Normal rest win rate:    {normal_wr:.3f}')
print(f'Penalty:                 {(normal_wr - b2b_wr)*100:.1f}% lower win rate on B2B nights')

print('\nRest day distribution (top values):')
print(team_games['days_rest'].value_counts().sort_index().head(10).to_string())

Rest Days + Back-to-Back computed

B2B games in dataset:    8,194
B2B win rate:            0.437
Normal rest win rate:    0.509
Penalty:                 7.3% lower win rate on B2B nights

Rest day distribution (top values):
days_rest
0.854167       1
0.875000       1
0.895833      35
0.916667     120
0.927083       1
0.937500     656
0.947917       1
0.958333    1258
0.979167    2064
0.989583       1


---
## Section 3 — Rebuild model_df + Baseline Check

Now we merge all new features (Elo + rest/B2B) into `model_df` (one row per game) and run a plain XGBoost with the same default params as Notebook 1.

This tells us **how much of the improvement comes from better features alone** before any model tuning.

In [5]:
# ── REBUILD model_df WITH ALL FEATURES ──────────────────────────────────────────
model_df = games[['gameId', 'gameDateTimeEst',
                  'hometeamCity', 'hometeamName',
                  'awayteamCity', 'awayteamName',
                  'home_win']].copy()
model_df['home_team'] = model_df['hometeamCity'] + ' ' + model_df['hometeamName']
model_df['away_team'] = model_df['awayteamCity'] + ' ' + model_df['awayteamName']
model_df = model_df.drop(columns=['hometeamCity', 'hometeamName', 'awayteamCity', 'awayteamName'])

# Original 5 rolling features per team
base_cols = ['gameId', 'teamName', 'rolling_win_rate', 'rolling_pts_scored',
             'rolling_pts_allowed', 'rolling_point_diff', 'win_streak']

home_base = team_games[base_cols].copy()
home_base.columns = ['gameId', 'home_team'] + ['home_' + c for c in base_cols[2:]]
away_base = team_games[base_cols].copy()
away_base.columns = ['gameId', 'away_team'] + ['away_' + c for c in base_cols[2:]]

model_df = model_df.merge(home_base, on=['gameId', 'home_team'], how='left')
model_df = model_df.merge(away_base, on=['gameId', 'away_team'], how='left')

# Elo features (game level)
model_df = model_df.merge(
    elo_df[['gameId', 'home_elo', 'away_elo', 'elo_diff', 'elo_win_prob']],
    on='gameId', how='left')

# Rest / B2B features (team level -> game level)
rest_cols = ['gameId', 'teamName', 'days_rest', 'is_back_to_back']
home_rest = team_games[rest_cols].copy()
home_rest.columns = ['gameId', 'home_team', 'home_days_rest', 'home_b2b']
away_rest = team_games[rest_cols].copy()
away_rest.columns = ['gameId', 'away_team', 'away_days_rest', 'away_b2b']

model_df = model_df.merge(home_rest, on=['gameId', 'home_team'], how='left')
model_df = model_df.merge(away_rest, on=['gameId', 'away_team'], how='left')
model_df = model_df.dropna().reset_index(drop=True)

print(f'model_df shape: {model_df.shape}')
print(f'Columns ({len(model_df.columns)}): {model_df.columns.tolist()}')

# ── CHRONOLOGICAL TRAIN / TEST SPLIT ────────────────────────────────────────────
SPLIT_DATE = pd.Timestamp('2020-01-01')
train_df = model_df[model_df['gameDateTimeEst'] <  SPLIT_DATE].reset_index(drop=True)
test_df  = model_df[model_df['gameDateTimeEst'] >= SPLIT_DATE].reset_index(drop=True)

ALL_FEATURE_COLS = [
    'home_rolling_win_rate', 'home_rolling_pts_scored', 'home_rolling_pts_allowed',
    'home_rolling_point_diff', 'home_win_streak',
    'away_rolling_win_rate', 'away_rolling_pts_scored', 'away_rolling_pts_allowed',
    'away_rolling_point_diff', 'away_win_streak',
    'elo_diff', 'elo_win_prob',
    'home_days_rest', 'away_days_rest',
    'home_b2b', 'away_b2b'
]

X_train = train_df[ALL_FEATURE_COLS]
y_train = train_df['home_win']
X_test  = test_df[ALL_FEATURE_COLS]
y_test  = test_df['home_win']

print(f'\nTrain: {len(train_df):,} games  '
      f'({train_df["gameDateTimeEst"].min().year}–{train_df["gameDateTimeEst"].max().year})')
print(f'Test:  {len(test_df):,} games  '
      f'({test_df["gameDateTimeEst"].min().year}–{test_df["gameDateTimeEst"].max().year})')
print(f'Features: {len(ALL_FEATURE_COLS)} (was 10 in notebook 1)')

model_df shape: (25543, 23)
Columns (23): ['gameId', 'gameDateTimeEst', 'home_win', 'home_team', 'away_team', 'home_rolling_win_rate', 'home_rolling_pts_scored', 'home_rolling_pts_allowed', 'home_rolling_point_diff', 'home_win_streak', 'away_rolling_win_rate', 'away_rolling_pts_scored', 'away_rolling_pts_allowed', 'away_rolling_point_diff', 'away_win_streak', 'home_elo', 'away_elo', 'elo_diff', 'elo_win_prob', 'home_days_rest', 'home_b2b', 'away_days_rest', 'away_b2b']

Train: 19,855 games  (2000–2019)
Test:  5,688 games  (2020–2026)
Features: 16 (was 10 in notebook 1)


In [7]:
# ── XGBoost v2: SAME PARAMS, NEW FEATURES — measures raw feature value ───────────
model_xgb_v2 = XGBClassifier(
    n_estimators=100, max_depth=4, learning_rate=0.1,
    random_state=42, verbosity=0, eval_metric='logloss'
)
model_xgb_v2.fit(X_train, y_train)
preds_v2  = model_xgb_v2.predict(X_test)
acc_xgb_v2 = accuracy_score(y_test, preds_v2)

print(f'XGBoost v2 (new features, same params): {acc_xgb_v2:.3f}')
print(f'XGBoost v1 (notebook 1):                0.618')
print(f'Raw feature gain:                       +{(acc_xgb_v2 - 0.618)*100:.1f}%')

# Feature importances
feat_imp = pd.Series(model_xgb_v2.feature_importances_, index=ALL_FEATURE_COLS)
feat_imp = feat_imp.sort_values(ascending=False)
print('\nTop 10 Feature Importances:')
for feat, imp in feat_imp.head(10).items():
    bar = '#' * int(imp * 200)
    print(f'  {feat:<35} {imp:.4f}  {bar}')

XGBoost v2 (new features, same params): 0.637
XGBoost v1 (notebook 1):                0.618
Raw feature gain:                       +1.9%

Top 10 Feature Importances:
  elo_win_prob                        0.6756  #######################################################################################################################################
  elo_diff                            0.1148  ######################
  home_rolling_point_diff             0.0262  #####
  away_rolling_point_diff             0.0221  ####
  away_days_rest                      0.0197  ###
  home_win_streak                     0.0175  ###
  home_days_rest                      0.0172  ###
  away_rolling_pts_scored             0.0163  ###
  home_rolling_win_rate               0.0159  ###
  home_rolling_pts_allowed            0.0157  ###


---
## Section 4 — Optuna Hyperparameter Tuning

**Why XGBoost default params are not the optimum:**
- `n_estimators=100` is often too low — more trees improve calibration
- `max_depth=4` might be too shallow or too deep depending on feature interactions
- `learning_rate=0.1` is a coarse starting point

**Why Optuna vs grid search:** Bayesian optimization builds a probabilistic model of the objective surface (using TPE — Tree of Parzen Estimators) and focuses new trials on regions that performed well before. With the same compute budget, it finds better solutions than grid search.

**Why TimeSeriesSplit for CV:** Standard k-fold would mix future games into the training fold — a form of data leakage. TimeSeriesSplit respects the temporal order of data.

In [8]:
# ── IMPROVEMENT 3: OPTUNA HYPERPARAMETER TUNING ──────────────────────────────────
if importlib.util.find_spec('optuna') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'optuna', '-q'])

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 200, 800),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma':            trial.suggest_float('gamma', 0.0, 3.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 5.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 5.0, log=True),
        'random_state': 42, 'verbosity': 0, 'eval_metric': 'logloss'
    }

    tscv = TimeSeriesSplit(n_splits=5)
    scores = []
    for tr_idx, val_idx in tscv.split(X_train):
        X_tr  = X_train.iloc[tr_idx];  X_val = X_train.iloc[val_idx]
        y_tr  = y_train.iloc[tr_idx];  y_val = y_train.iloc[val_idx]
        m = XGBClassifier(**params)
        m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        scores.append(accuracy_score(y_val, m.predict(X_val)))
    return np.mean(scores)

print('Running Optuna (50 trials, TPE sampler)...')
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)
study.optimize(objective, n_trials=50, show_progress_bar=True)

best_params = study.best_params.copy()
best_params.update({'random_state': 42, 'verbosity': 0, 'eval_metric': 'logloss'})

print(f'\nBest CV accuracy: {study.best_value:.4f}')
print('\nBest parameters found:')
for k, v in study.best_params.items():
    print(f'  {k:<25} {v}')

# Train on full training set with best params
model_xgb_opt = XGBClassifier(**best_params)
model_xgb_opt.fit(X_train, y_train)
preds_opt   = model_xgb_opt.predict(X_test)
acc_xgb_opt = accuracy_score(y_test, preds_opt)

print(f'\nOptuna XGBoost test accuracy:    {acc_xgb_opt:.3f}')
print(f'XGBoost v2 (default params):     {acc_xgb_v2:.3f}')
print(f'XGBoost v1 (notebook 1):         0.618')
print(f'Tuning gain over v2:             +{(acc_xgb_opt - acc_xgb_v2)*100:.1f}%')

Running Optuna (50 trials, TPE sampler)...


  0%|          | 0/50 [00:00<?, ?it/s]


Best CV accuracy: 0.6673

Best parameters found:
  n_estimators              215
  max_depth                 7
  learning_rate             0.0315231965521318
  subsample                 0.86500683939344
  colsample_bytree          0.8561711433906498
  min_child_weight          6
  gamma                     2.5887819655597584
  reg_alpha                 4.8713386814938096
  reg_lambda                3.171112652626632

Optuna XGBoost test accuracy:    0.635
XGBoost v2 (default params):     0.637
XGBoost v1 (notebook 1):         0.618
Tuning gain over v2:             +-0.1%


---
## Section 5 — Attention-LSTM

**Problem with the standard LSTM from Notebook 1:** The LSTM's final hidden state aggregates all 10 timesteps with equal implicit weight. A game from 10 games ago is treated the same as yesterday's game.

**Bahdanau Attention fix:** A learned attention layer sits on top of the LSTM's output sequence. It produces a weight for each timestep, then computes a weighted sum. The model can learn that recent games should dominate — and it can also learn to down-weight noisy games (blowouts, opponent rest mismatches).

**New feature set:** 7 features per team (added `days_rest` + `team_elo`) × 2 teams = **14 features per timestep** → input shape `(N, 10, 14)`

In [9]:
# ── IMPROVEMENT 4: ATTENTION-LSTM WITH NEW FEATURES ─────────────────────────────

SEQ_FEATURE_COLS = [
    'rolling_win_rate', 'rolling_pts_scored', 'rolling_pts_allowed',
    'rolling_point_diff', 'win_streak',
    'days_rest',   # NEW
    'team_elo'     # NEW
]

def build_sequences(team_games_df, game_df, feature_cols, window=10):
    scaler = StandardScaler()
    tg = team_games_df.copy().dropna(subset=feature_cols)
    tg = tg.sort_values(['teamName', 'date'])
    tg[feature_cols] = scaler.fit_transform(tg[feature_cols])

    # Build dict: (gameId, teamName) -> sequence of last `window` games
    team_seqs = {}
    for team, grp in tg.groupby('teamName'):
        grp = grp.reset_index(drop=True)
        if len(grp) < window + 1:
            continue
        vals     = grp[feature_cols].values
        game_ids = grp['gameId'].values
        for i in range(window, len(grp)):
            team_seqs[(game_ids[i], team)] = vals[i - window:i]  # strictly past

    # Match sequences to game-level dataframe
    X, y, dates = [], [], []
    for _, row in game_df.iterrows():
        h_key = (row['gameId'], row['home_team'])
        a_key = (row['gameId'], row['away_team'])
        if h_key in team_seqs and a_key in team_seqs:
            combined = np.concatenate([team_seqs[h_key], team_seqs[a_key]], axis=1)
            X.append(combined)
            y.append(row['home_win'])
            dates.append(row['gameDateTimeEst'])

    return np.array(X), np.array(y), dates

X_all_seq, y_all_seq, dates_seq = build_sequences(
    team_games, model_df, SEQ_FEATURE_COLS, window=10
)

split_mask    = np.array([d < SPLIT_DATE for d in dates_seq])
X_train_seq   = X_all_seq[split_mask]
y_train_seq   = y_all_seq[split_mask]
X_test_seq    = X_all_seq[~split_mask]
y_test_seq    = y_all_seq[~split_mask]

n_timesteps = X_train_seq.shape[1]
n_features  = X_train_seq.shape[2]

print(f'Sequence shapes — Train: {X_train_seq.shape}, Test: {X_test_seq.shape}')
print(f'Input: {n_timesteps} timesteps x {n_features} features '
      f'({len(SEQ_FEATURE_COLS)} home + {len(SEQ_FEATURE_COLS)} away)')

# ── BAHDANAU ATTENTION LAYER ─────────────────────────────────────────────────────
class BahdanauAttention(Layer):
    """Soft attention over LSTM output sequence."""
    def __init__(self, units=32, **kwargs):
        super().__init__(**kwargs)
        self.W = Dense(units)
        self.V = Dense(1)

    def call(self, lstm_output):
        # lstm_output: (batch, timesteps, hidden)
        score   = self.V(tf.nn.tanh(self.W(lstm_output)))  # (batch, timesteps, 1)
        weights = tf.nn.softmax(score, axis=1)              # (batch, timesteps, 1)
        context = weights * lstm_output                     # (batch, timesteps, hidden)
        return tf.reduce_sum(context, axis=1)               # (batch, hidden)

    def get_config(self):
        return super().get_config()

# ── BUILD + TRAIN ATTENTION-LSTM ─────────────────────────────────────────────────
tf.random.set_seed(42)

inputs  = Input(shape=(n_timesteps, n_features))
x       = LSTM(64, return_sequences=True)(inputs)
x       = Dropout(0.3)(x)
x       = BahdanauAttention(units=32)(x)   # weighted sum replaces 2nd LSTM
x       = Dense(32, activation='relu')(x)
x       = Dropout(0.2)(x)
x       = Dense(16, activation='relu')(x)
outputs = Dense(1, activation='sigmoid')(x)

model_attn = Model(inputs, outputs)
model_attn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_attn.summary()

early_stop = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)

history = model_attn.fit(
    X_train_seq, y_train_seq,
    epochs=60,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

_, acc_attn = model_attn.evaluate(X_test_seq, y_test_seq, verbose=0)
attn_probs  = model_attn.predict(X_test_seq, verbose=0).flatten()

print(f'\nAttention-LSTM accuracy:  {acc_attn:.3f}')
print(f'Original LSTM (nb1):      0.629')
print(f'Gain:                    +{(acc_attn - 0.629)*100:.1f}%')

Sequence shapes — Train: (19699, 10, 14), Test: (5688, 10, 14)
Input: 10 timesteps x 14 features (7 home + 7 away)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 10, 14)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 10, 64)         │        20,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 10, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bahdanau_attention              │ (None, 64)             │         2,113 │
│ (BahdanauAttention)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,962 (97.51 KB)

 Trainable params: 24,962 (97.51 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/60
247/247 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.6577 - loss: 0.6198 - val_accuracy: 0.6642 - val_loss: 0.6091
Epoch 2/60
247/247 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.6650 - loss: 0.6104 - val_accuracy: 0.6670 - val_loss: 0.6074
Epoch 3/60
247/247 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.6648 - loss: 0.6087 - val_accuracy: 0.6668 - val_loss: 0.6087
Epoch 4/60
247/247 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.6672 - loss: 0.6088 - val_accuracy: 0.6688 - val_loss: 0.6078
Epoch 5/60
247/247 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.6665 - loss: 0.6072 - val_accuracy: 0.6668 - val_loss: 0.6079
Epoch 6/60
247/247 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.6705 - loss: 0.6061 - val_accuracy: 0.6657 - val_loss: 0.6082
Epoch 7/60
247/247 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.6697 - loss: 0.6058 - val_accuracy: 0.6647 - val_loss: 0.6086
Epoch 8/60
247/247 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.6693 - loss: 0.6050 - val_accuracy:

---
## Section 6 — Proper OOF Stacking (Fixing Notebook 1 Leakage)

**The bug in Notebook 1:**
```python
meta_model.fit(meta_X_test, y_test)   # ← trained ON test data
meta_preds = meta_model.predict(meta_X_test)  # ← predicted on SAME test data
meta_accuracy = accuracy_score(y_test, meta_preds)  # ← evaluated on training data!
```
The meta-learner memorized `y_test`. This inflated the reported 63.1%.

**The proper fix — Out-of-Fold (OOF) stacking:**
1. Split training data into 5 time-ordered folds
2. For each fold: train base models on the past, predict the future fold → OOF predictions
3. After all folds: have honest predictions for the entire training set
4. Train meta-learner on those OOF predictions (it never saw test data)
5. Retrain base models on full training set → predict test set
6. Meta-learner predicts test set → evaluate honestly

In [10]:
# ── IMPROVEMENT 5: PROPER OOF STACKING ──────────────────────────────────────────
N_SPLITS = 5
tscv     = TimeSeriesSplit(n_splits=N_SPLITS)

oof_xgb = np.zeros(len(X_train))
oof_lr  = np.zeros(len(X_train))

print(f'Generating OOF predictions via {N_SPLITS}-fold TimeSeriesSplit...')
print(f'(Folds respect chronological order — no future leakage)\n')

for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train)):
    X_tr  = X_train.iloc[tr_idx];  X_val = X_train.iloc[val_idx]
    y_tr  = y_train.iloc[tr_idx];  y_val = y_train.iloc[val_idx]

    # XGBoost fold
    xgb_f = XGBClassifier(**best_params)
    xgb_f.fit(X_tr, y_tr)
    oof_xgb[val_idx] = xgb_f.predict_proba(X_val)[:, 1]

    # Logistic Regression fold
    lr_f = LogisticRegression(max_iter=1000)
    lr_f.fit(X_tr, y_tr)
    oof_lr[val_idx] = lr_f.predict_proba(X_val)[:, 1]

    fold_xgb_acc = accuracy_score(y_val, (oof_xgb[val_idx] > 0.5).astype(int))
    fold_lr_acc  = accuracy_score(y_val, (oof_lr[val_idx]  > 0.5).astype(int))
    yr_start     = train_df['gameDateTimeEst'].iloc[val_idx[0]].year
    yr_end       = train_df['gameDateTimeEst'].iloc[val_idx[-1]].year
    print(f'  Fold {fold+1} ({yr_start}–{yr_end}, {len(val_idx):,} games)  '
          f'XGB: {fold_xgb_acc:.3f}  LR: {fold_lr_acc:.3f}')

# Train meta-learner on OOF predictions (no leakage into test set)
meta_X_train_oof = np.column_stack([oof_xgb, oof_lr])
meta_model_oof   = LogisticRegression(max_iter=1000)
meta_model_oof.fit(meta_X_train_oof, y_train)
print('\nMeta-learner (LogReg) trained on OOF predictions')
print(f'Meta-learner weights — XGB: {meta_model_oof.coef_[0][0]:.3f}, '
      f'LR: {meta_model_oof.coef_[0][1]:.3f}')

# Retrain base models on FULL training set
xgb_full = XGBClassifier(**best_params)
xgb_full.fit(X_train, y_train)
test_xgb_probs = xgb_full.predict_proba(X_test)[:, 1]

lr_full = LogisticRegression(max_iter=1000)
lr_full.fit(X_train, y_train)
test_lr_probs = lr_full.predict_proba(X_test)[:, 1]

# Meta-learner predicts test set
meta_X_test_oof  = np.column_stack([test_xgb_probs, test_lr_probs])
stacked_preds_oof = meta_model_oof.predict(meta_X_test_oof)
acc_oof_stack     = accuracy_score(y_test, stacked_preds_oof)

acc_xgb_full = accuracy_score(y_test, xgb_full.predict(X_test))
acc_lr_full  = accuracy_score(y_test, lr_full.predict(X_test))

print(f'\n--- OOF Stacking Results ---')
print(f'XGBoost (retrained full):    {acc_xgb_full:.3f}')
print(f'Logistic Regression (full):  {acc_lr_full:.3f}')
print(f'OOF Stacked Ensemble:        {acc_oof_stack:.3f}')
print(f'\nThis is an honest accuracy — meta-learner never saw y_test during training.')

Generating OOF predictions via 5-fold TimeSeriesSplit...
(Folds respect chronological order — no future leakage)

  Fold 1 (2002–2005, 3,309 games)  XGB: 0.653  LR: 0.665
  Fold 2 (2005–2009, 3,309 games)  XGB: 0.664  LR: 0.667
  Fold 3 (2009–2012, 3,309 games)  XGB: 0.672  LR: 0.682
  Fold 4 (2012–2016, 3,309 games)  XGB: 0.679  LR: 0.682
  Fold 5 (2016–2019, 3,309 games)  XGB: 0.668  LR: 0.676

Meta-learner (LogReg) trained on OOF predictions
Meta-learner weights — XGB: -2.306, LR: 3.697

--- OOF Stacking Results ---
XGBoost (retrained full):    0.635
Logistic Regression (full):  0.642
OOF Stacked Ensemble:        0.575

This is an honest accuracy — meta-learner never saw y_test during training.


---
## Section 7 — Final Accuracy Summary

All models, in order of development, with delta vs the 55.3% baseline.

In [15]:
# ── FINAL ACCURACY SUMMARY ───────────────────────────────────────────────────────
BASELINE = 0.553

results = [
    ('Baseline (always predict home win)',   0.553),
    ('ARIMA (score regression)',             0.511),
    ('Logistic Regression v1',               0.614),
    ('XGBoost v1  [Notebook 1]',             0.618),
    ('CNN-1D      [Notebook 1]',             0.620),
    ('LSTM v1     [Notebook 1]',             0.629),
    ('Ensemble v1 [Notebook 1 — LEAKY]',     0.631),
    ('XGBoost v2  [+ Elo + Rest/B2B]',       acc_xgb_v2),
    ('XGBoost v3  [Optuna-tuned]',           acc_xgb_opt),
    ('Attention-LSTM [+ Elo + Rest/B2B]',    acc_attn),
    ('OOF Stack  [XGB + LR, clean]',         acc_oof_stack),
]

best_acc  = max(r[1] for r in results)
best_name = [r[0] for r in results if r[1] == best_acc][0]

print('=' * 68)
print(f'{"MODEL":<45} {"ACC":>6}  {"DELTA":>7}  {"BAR"}')
print('=' * 68)
for name, acc in results:
    delta   = acc - BASELINE
    bar     = '#' * int(abs(delta) * 100)
    marker  = ' <- BEST' if name == best_name else ''
    sign    = '+' if delta >= 0 else ''
    print(f'{name:<45} {acc:.3f}  {sign}{delta:.3f}   {bar}{marker}')
print('=' * 68)

print(f'\nBest accuracy:            {best_acc:.3f}  ({best_name})')
print(f'Improvement over baseline: +{(best_acc - BASELINE)*100:.1f}%')
print(f'Gap to 70% target:         {(0.70 - best_acc)*100:.1f}%')
print()
print('Next step to close the gap:')
print('  Add player availability features from PlayerBoxScores.csv')
print('  (is the team\'s top scorer playing tonight?)')
print('  Expected gain: +2–3.5%  —  this is the gate to 70%')

MODEL                                            ACC    DELTA  BAR
Baseline (always predict home win)            0.553  +0.000   
ARIMA (score regression)                      0.511  -0.042   ####
Logistic Regression v1                        0.614  +0.061   ######
XGBoost v1  [Notebook 1]                      0.618  +0.065   ######
CNN-1D      [Notebook 1]                      0.620  +0.067   ######
LSTM v1     [Notebook 1]                      0.629  +0.076   #######
Ensemble v1 [Notebook 1 — LEAKY]              0.631  +0.078   #######
XGBoost v2  [+ Elo + Rest/B2B]                0.637  +0.084   ######## <- BEST
XGBoost v3  [Optuna-tuned]                    0.635  +0.082   ########
Attention-LSTM [+ Elo + Rest/B2B]             0.629  +0.076   #######
OOF Stack  [XGB + LR, clean]                  0.575  +0.022   ##

Best accuracy:            0.637  (XGBoost v2  [+ Elo + Rest/B2B])
Improvement over baseline: +8.4%
Gap to 70% target:         6.3%

Next step to close the gap:
  Add play